In [7]:
import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection
from matplotlib.colors import ListedColormap
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

#O recreere a articolului https://www.mdpi.com/2673-9909/5/3/85
#A recreation of https://www.mdpi.com/2673-9909/5/3/85 article
alpha = 0.1
beta = 0.5
gamma = 0.001
rows = 100
cols = 100
R = 1.0
ring_fraction = 0.92
hole_fraction = 0.10
n_random_seeds = 25

n_steps = 900
save_every = 15

seed = 1
rng = np.random.default_rng(seed)

N, B, F, E = 0, 1, 2, 3

In [8]:
X = np.zeros((rows, cols))
Y = np.zeros((rows, cols))

for j in range(rows):
    for i in range(cols):
        X[j, i] = i*math.sqrt(3)*R
        if j % 2 == 1:
            X[j, i] += math.sqrt(3)/2*R
        Y[j, i] = j * 1.5 * R

hexagon_corners = []

for j in range(rows):
    for i in range(cols):
        corners = []
        for k in range(6):
            angle = math.pi/3*k+math.pi/6
            corners.append((X[j, i] + R * math.cos(angle),
                            Y[j, i] + R * math.sin(angle)))
        hexagon_corners.append(corners)

In [9]:
even_of = [(0, -1), (0, 1), (-1, -1), (-1, 0), (1, -1), (1, 0)]
odd_of = [(0, -1), (0, 1), (-1, 0), (-1, 1), (1, 0), (1, 1)]
neighbours = [[[] for i in range(cols)] for j in range(rows)]
for j in range(rows):
    for i in range(cols):
        offsets = even_of if j % 2 == 0 else odd_of
        for d_row, d_col in offsets:
            nj = j + d_row
            ni = i + d_col

            if 0 <= nj < rows and 0 <= ni < cols:
                neighbours[j][i].append((nj, ni))

In [10]:
state = np.zeros((rows, cols), dtype=int)

center_x = X.mean()
center_y = Y.mean()
half_width = 0.46 * X.max()
half_height = 0.46 * Y.max() * 0.8
hole_radius = hole_fraction * half_width * 2

for j in range(rows):
    for i in range(cols):
        ellipse_value = ((X[j, i] - center_x) / half_width) ** 2 + ((Y[j, i] - center_y) / half_height) ** 2
        distance_to_center = math.hypot(X[j, i] - center_x, Y[j, i] - center_y)

        if ellipse_value > 1 or distance_to_center <= hole_radius:
            state[j, i] = E
        elif ellipse_value > ring_fraction ** 2:
            state[j, i] = F
        else:
            state[j, i] = N

for j in range(rows):
    for i in range(cols):
        distance_to_center = math.hypot(X[j, i] - center_x, Y[j, i] - center_y)

        if state[j, i] == N and distance_to_center <= hole_radius + 1.6 * R and j % 2 == 0:
            state[j, i] = F

empty_cells = [(j, i) for j in range(rows) for i in range(cols) if state[j, i] == N]
chosen = rng.choice(len(empty_cells), n_random_seeds, replace=False)

for index in chosen:
    j, i = empty_cells[index]
    state[j, i] = F

u = np.zeros((rows, cols))
v = np.zeros((rows, cols))

for j in range(rows):
    for i in range(cols):
        if state[j, i] == N or state[j, i] == E:
            u[j, i] = beta
        if state[j, i] == F:
            v[j, i] = 1.0

In [11]:
def average_u(u, j, i):
    total = 0.0
    for nj, ni in neighbours[j][i]:
        total += u[nj, ni]
    return total / 6.0


def make_borders(state, u, v):
    for j in range(1, rows - 1):
        for i in range(1, cols - 1):
            if state[j, i] != N:
                continue

            for nj, ni in neighbours[j][i]:
                if state[nj, ni] == F:
                    state[j, i] = B
                    v[j, i] = u[j, i]
                    u[j, i] = 0.0
                    break


def step(state, u, v):
    new_state = state.copy()
    new_u = u.copy()
    new_v = v.copy()

    for j in range(1, rows - 1):
        for i in range(1, cols - 1):
            if state[j, i] == N:
                mean_u = average_u(u, j, i)
                new_u[j, i] = u[j, i] + (2 * alpha / 3) * (mean_u - u[j, i])

            elif state[j, i] == B:
                mean_u = average_u(u, j, i)
                new_v[j, i] = v[j, i] + gamma + (2 * alpha / 3) * mean_u

                if new_v[j, i] >= 1.0:
                    new_state[j, i] = F

    make_borders(new_state, new_u, new_v)
    return new_state, new_u, new_v


make_borders(state, u, v)

In [12]:
import time

start_time = time.time()
history = [state.copy()]
saved_steps = [0]
solid_fraction = []

inside = (state != E).sum()
solid_fraction.append((state == F).sum() / inside)

for t in range(1, n_steps + 1):
    state, u, v = step(state, u, v)

    if t % save_every == 0:
        history.append(state.copy())
        saved_steps.append(t)
        solid_fraction.append((state == F).sum() / inside)

In [13]:
colours = ListedColormap(["lightskyblue", "orange", "dimgray", "white"])
plt.rcParams["animation.embed_limit"] = 200

fig, ax = plt.subplots(figsize=(5, 5), dpi=70)

hexes = PolyCollection(
    hexagon_corners,
    cmap=colours,
    edgecolors="black",
    linewidths=0.1
)

hexes.set_array(history[0].ravel())
hexes.set_clim(-0.5, 3.5)

ax.add_collection(hexes)
ax.autoscale_view()
ax.set_aspect("equal")
ax.axis("off")

title = ax.set_title("")


def draw_frame(f):
    hexes.set_array(history[f].ravel())
    title.set_text(f"step {saved_steps[f]}")
    return hexes, title


anim = FuncAnimation(fig, draw_frame, frames=len(history), interval=80)
html = anim.to_jshtml()

plt.close(fig)

HTML(html)